In [47]:
"""
This file builds the RAG system.

Steps happening here:
1. Load policy document
2. Split into chunks
3. Convert chunks into embeddings
4. Store embeddings in vector database
5. Create retriever function to query policies
"""

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [48]:
# -----------------------------------------
# Load loan policy document
# -----------------------------------------

loader = TextLoader("data/loaneligibility-policy.txt")
documents = loader.load()

In [49]:
# -----------------------------------------
# Split document into chunks
# -----------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

docs = splitter.split_documents(documents)

In [50]:
# -----------------------------------------
# Convert chunks into embeddings
# -----------------------------------------

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# -----------------------------------------
# Store embeddings into vector DB
# -----------------------------------------

vectorstore = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="loan_policy_db"
)

vectorstore.persist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
# -----------------------------------------
# Retriever function used by agents
# -----------------------------------------

def query_policies(question: str) -> str:
    """
    This function retrieves relevant policy rules from the vector DB.

    Input:
        question -> what we want to know about policies

    Output:
        concatenated policy rules text
    """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    docs = retriever.invoke(question)

    return "\n".join([d.page_content for d in docs])

In [52]:

# quick test
if __name__ == "__main__":
    result = query_policies("loan eligibility rule")
    print(result)

Eligibility status

Maximum possible loan

Suitable loan plan
Eligibility status

Maximum possible loan

Suitable loan plan
Bank Loan Eligibility & Policy Knowledge Base
1. Overview


In [53]:
docs = vectorstore.get()
rules=''''''
for i, doc in enumerate(docs["documents"]):
    # print(f"\nChunk {i+1}")
    rules+=doc
rules
    

'Loan Policy Rules\n\n1. Minimum salary required for personal loan is 25,000 per month.\n\n2. Minimum credit score required is 700.\n\n3. Maximum Debt To Income (DTI) ratio allowed is 55%.4. Maximum loan amount allowed is 20 times the monthly salary.\n\n5. Interest rate typically ranges between 9% to 13%.\n\n6. Maximum loan tenure allowed is 7 years.7. Existing EMI obligations should not exceed 40% of salary.\n\n8. Applicants with credit score above 750 get priority approval.\n\n9. Applicants with credit score below 650 are considered high risk.10. Salary stability of at least 6 months preferred.Loan Policy Rules\n\n1. Minimum salary required for personal loan is 25,000 per month.\n\n2. Minimum credit score required is 700.\n\n3. Maximum Debt To Income (DTI) ratio allowed is 55%.4. Maximum loan amount allowed is 20 times the monthly salary.\n\n5. Interest rate typically ranges between 9% to 13%.\n\n6. Maximum loan tenure allowed is 7 years.7. Existing EMI obligations should not exceed 

In [54]:
"""
Agentic Loan Eligibility Assistant using LangGraph + RAG

Workflow:

User Input
   ↓
collect_financials
   ↓
check_eligibility
   ↓
predict_approval_chance
   ↓
suggest_loan_plan
   ↓
review_response
   ↓
END
"""
import json
import re
from typing import Optional

# Pydantic & LangGraph
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

# Core logic
from langchain_core.prompts import PromptTemplate

# This requires the 'langchain' package
# from langchain.output_parsers import StructuredOutputParser, ResponseSchema
# This requires the 'langchain-ollama' package
from langchain_ollama import ChatOllama

from langchain_community.chat_models import ChatOllama

# initialize the LLM model
llm = ChatOllama(model="llama3.2")
# =====================================================
# 1️⃣ State Definition
# =====================================================

class LoanState(BaseModel):
    """
    This class stores the information flowing through the graph.

    LangGraph passes this state between nodes.
    Each node can read/update values.
    """

    user_input: str

    salary: Optional[int] = Field(default=None)
    credit_score: Optional[int] = Field(default=None)
    loan_amount: Optional[int] = Field(default=None)
    tenure_years: Optional[int] = Field(default=None)
    existing_emi: Optional[int] = Field(default=None)

    retrieved_rules: Optional[str] = None
    eligibility_status: Optional[str] = None
    approval_score: Optional[float] = None
    suggested_plan: Optional[str] = None
    final_reply: Optional[str] = None


# =====================================================
# 2️⃣ Node 1 — Extract financial details
# =====================================================

def collect_financials(state: LoanState):
    """
    Extracts salary, credit score, loan amount, tenure, EMI
    from user text using regex.
    """

    text = state.user_input.lower()

    # extract salary
    salary_match = re.search(r'(\d+)\s?k', text)
    if salary_match:
        state.salary = int(salary_match.group(1)) * 1000

    # extract credit score
    credit_match = re.search(r'credit score\s*(\d+)', text)
    if credit_match:
        state.credit_score = int(credit_match.group(1))

    # extract loan amount
    loan_match = re.search(r'(\d+)\s?lakhs?', text)
    if loan_match:
        state.loan_amount = int(loan_match.group(1)) * 100000

    # extract tenure
    tenure_match = re.search(r'(\d+)\s?yrs?', text)
    if tenure_match:
        state.tenure_years = int(tenure_match.group(1))

    # extract existing EMI
    emi_match = re.search(r'emi\s*(\d+)k', text)
    if emi_match:
        state.existing_emi = int(emi_match.group(1)) * 1000

    return state


# =====================================================
# 3️⃣ Node 2 — Eligibility Check using RAG
# =====================================================

import json

def check_eligibility(state: LoanState):
    """
    Evaluates loan eligibility based on the Bank Loan Policy Knowledge Base.

    Steps implemented exactly as defined in the policy:

    1. Retrieve policies using RAG
    2. Check general eligibility (credit score category)
    3. Validate EMI burden rule
    4. Apply loan-type specific rules
    5. Determine eligibility and max loan amount
    """

    # ------------------------------------------------
    # Step 1 — Retrieve policies using RAG
    # ------------------------------------------------
    rules = query_policies("bank loan eligibility rules")
    state.retrieved_rules = rules

    salary = state.salary
    credit = state.credit_score
    emi = state.existing_emi or 0
    loan_amount = state.loan_amount
    tenure = state.tenure_years

    eligible = True
    reasons = []
    max_loan = None

    # ------------------------------------------------
    # Step 2 — Credit Score Category
    # ------------------------------------------------

    if credit < 600:
        eligible = False
        reasons.append("Credit score below 600 (very high risk)")
    elif 600 <= credit <= 649:
        reasons.append("High risk credit profile")
    elif 650 <= credit <= 699:
        reasons.append("Moderate credit profile")
    elif 700 <= credit <= 749:
        reasons.append("Good credit profile")
    else:
        reasons.append("Excellent credit profile")

    # ------------------------------------------------
    # Step 3 — EMI Burden Ratio
    # ------------------------------------------------

    if salary:
        emi_ratio = (emi / salary) * 100

        if emi_ratio > 50:
            eligible = False
            reasons.append("EMI ratio exceeds 50% limit")
        elif emi_ratio > 40:
            reasons.append("Moderate EMI burden")
        else:
            reasons.append("Safe EMI burden")

    # ------------------------------------------------
    # Step 4 — Determine Loan Type
    # (simple detection from user text)
    # ------------------------------------------------

    text = state.user_input.lower()

    if "home" in text:
        loan_type = "home"
    elif "car" in text:
        loan_type = "car"
    elif "education" in text:
        loan_type = "education"
    else:
        loan_type = "personal"

    # ------------------------------------------------
    # Step 5 — Apply Loan Type Rules
    # ------------------------------------------------

    if loan_type == "personal":

        if salary < 30000:
            eligible = False
            reasons.append("Personal loan requires salary ≥ ₹30,000")

        if credit >= 750:
            max_loan = 2000000
        elif credit >= 700:
            max_loan = 1500000
        elif credit >= 650:
            max_loan = 1000000
        else:
            eligible = False

        if tenure and (tenure < 1 or tenure > 5):
            reasons.append("Personal loan tenure must be 1–5 years")

    elif loan_type == "home":

        if salary < 50000:
            eligible = False
            reasons.append("Home loan requires salary ≥ ₹50,000")

        if salary < 75000:
            max_loan = 3000000
        elif salary < 150000:
            max_loan = 6000000
        else:
            max_loan = 12000000

    elif loan_type == "car":

        if salary < 25000:
            eligible = False
            reasons.append("Car loan requires salary ≥ ₹25,000")

        if salary < 50000:
            max_loan = 700000
        elif salary < 100000:
            max_loan = 1200000
        else:
            max_loan = 2000000

    elif loan_type == "education":

        if credit < 650:
            eligible = False
            reasons.append("Education loan requires credit score ≥ 650")

        max_loan = 2000000

    # ------------------------------------------------
    # Step 6 — Validate Requested Loan Amount
    # ------------------------------------------------

    if max_loan and loan_amount and loan_amount > max_loan:
        reasons.append(
            f"Requested loan exceeds allowed maximum of ₹{max_loan:,}"
        )

    # ------------------------------------------------
    # Step 7 — Final Eligibility Decision
    # ------------------------------------------------

    if eligible:
        state.eligibility_status = "Eligible ✔"
    else:
        state.eligibility_status = "Not Eligible ❌"

    return state

# =====================================================
# 4️⃣ Node 3 — Predict approval chance
# =====================================================

def predict_approval_chance(state: LoanState):
    """
    Simple heuristic scoring system.

    Factors used:
    - credit score
    - salary
    - DTI
    """

    score = 50

    if state.credit_score:
        score += (state.credit_score - 650) * 0.2

    if state.salary and state.loan_amount:
        loan_ratio = state.loan_amount / (state.salary * 12)

        if loan_ratio < 5:
            score += 10
        elif loan_ratio > 10:
            score -= 10

    if state.salary and state.existing_emi:
        dti = state.existing_emi / state.salary

        if dti < 0.3:
            score += 10
        else:
            score -= 10

    score = max(0, min(100, score))

    state.approval_score = round(score, 2)

    return state


# =====================================================
# 5️⃣ Node 4 — Suggest loan plan
# =====================================================

def suggest_loan_plan(state: LoanState):
    """
    Suggests EMI plan.

    EMI formula used:
    EMI = P * r * (1+r)^n / ((1+r)^n - 1)
    """

    if not state.loan_amount or not state.tenure_years:
        return state

    P = state.loan_amount

    annual_rate = 0.10
    r = annual_rate / 12

    n = state.tenure_years * 12

    emi = (P * r * (1 + r)**n) / ((1 + r)**n - 1)

    emi = int(emi)

    state.suggested_plan = f"""
Loan Amount: ₹{P:,}
Tenure: {state.tenure_years} years
Estimated EMI: ₹{emi:,}
Interest Assumption: 10%
"""

    return state


# =====================================================
# 6️⃣ Node 5 — Final response formatting
# =====================================================

def review_response(state: LoanState):
    """
    Formats the final clean answer for the user.
    """

    state.final_reply = f"""
Loan Eligibility Result
------------------------

Eligibility Status: {state.eligibility_status}

Approval Probability: {state.approval_score}%

Recommended Loan Plan:
{state.suggested_plan}

Advice:
Maintain a strong credit score and keep EMI obligations low.
"""

    return state


# =====================================================
# 7️⃣ Build LangGraph Workflow
# =====================================================

workflow = StateGraph(LoanState)

workflow.add_node("collect_financials", collect_financials)
workflow.add_node("check_eligibility", check_eligibility)
workflow.add_node("predict_approval_chance", predict_approval_chance)
workflow.add_node("suggest_loan_plan", suggest_loan_plan)
workflow.add_node("review_response", review_response)


workflow.add_edge("collect_financials", "check_eligibility")
workflow.add_edge("check_eligibility", "predict_approval_chance")
workflow.add_edge("predict_approval_chance", "suggest_loan_plan")
workflow.add_edge("suggest_loan_plan", "review_response")
workflow.add_edge("review_response", END)


workflow.set_entry_point("collect_financials")

loan_app = workflow.compile()



In [57]:

# =====================================================
# 8️⃣ Example Run
# =====================================================

if __name__ == "__main__":

    query = "I earn 65k/month, credit score 740, loan required 10 lakhs for 3 yrs. Already paying emi 12k."

    result = loan_app.invoke({"user_input": query})

    print(result["final_reply"])


Loan Eligibility Result
------------------------

Eligibility Status: Eligible ✔

Approval Probability: 88.0%

Recommended Loan Plan:

Loan Amount: ₹1,000,000
Tenure: 3 years
Estimated EMI: ₹32,267
Interest Assumption: 10%


Advice:
Maintain a strong credit score and keep EMI obligations low.

